# 第三周教师演示：使用NumPy进行粮食产量分析
先讲知识准备，再发布学生综合任务。正式任务答案待学生尝试后展示。数据口径及来源见05_数据与来源说明.md。


## 01 启动与先修检查
适用第三周，依赖Python 3与NumPy。按顺序运行；输出到屏幕，最后导出只写outputs。先口答：[10, 20, 30][1]是多少？列表切片[1:3]取几项？函数参数写在何处？答案20、两项、圆括号内。


In [1]:
from pathlib import Path
import numpy as np
# Notebook请在本备课包文件夹启动；脚本以自身所在目录为基准。
BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA = BASE / "data" / "粮食产量年度数据.csv"
print("NumPy版本：", np.__version__)
print("数据文件存在：", DATA.exists())


NumPy版本： 2.3.5
数据文件存在： True


## 02 从列表到数组
教学模拟产量，单位万吨。列表乘2是重复；数组乘2是逐项相乘。import导入库，as np起短名称。


In [2]:
values = [100, 120, 110]
a = np.array(values, dtype=float)
print("列表乘2：", values * 2)
print("数组乘2：", a * 2)
print("shape、ndim、size、dtype：", a.shape, a.ndim, a.size, a.dtype)
print("平均产量：", np.mean(a))


列表乘2： [100, 120, 110, 100, 120, 110]
数组乘2： [200. 240. 220.]
shape、ndim、size、dtype： (3,) 1 3 float64
平均产量： 110.0


## 03 创建二维数组与形状
模拟表：行是年份，列是甲、乙地区产量（同为万吨）。shape是行列数，ndim是轴的个数。reshape只能改变摆放，元素总数不变。arange不含终点。


In [3]:
region = np.array([[10, 20], [30, 40], [50, 60]])
print(region)
print(region.shape, region.ndim, region.size)
print(np.arange(2014, 2024))
print(np.arange(6).reshape(3, 2))
print(np.zeros((2, 3)))
print(np.ones((2, 3)))


[[10 20]
 [30 40]
 [50 60]]
(3, 2) 2 6
[2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]
[[0 1]
 [2 3]
 [4 5]]
[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1. 1.]
 [1. 1. 1.]]


## 04 索引、切片与配对
a[行,列]，冒号表示全部；从0计数，切片右端不取。年份和产量必须同步取位置。


In [4]:
demo_years = np.array([2021, 2022, 2023])
demo_yields = np.array([100.0, 120.0, 110.0])
table = np.column_stack((demo_years, demo_yields))
print(table)
print("第一行第二列：", table[0, 1])
print("所有产量：", table[:, 1])
print("前两年：", table[:2, :])
print("最后一年：", demo_years[-1], demo_yields[-1])
print("转置后形状：", table.T.shape)


[[2021.  100.]
 [2022.  120.]
 [2023.  110.]]
第一行第二列： 100.0
所有产量： [100. 120. 110.]
前两年： [[2021.  100.]
 [2022.  120.]]
最后一年： 2023 110.0
转置后形状： (2, 3)


## 05 axis：哪个维度被汇总
只对同单位产量表求和，绝不把年份与产量相加。axis=0消去行的维度，得到每列合计；axis=1消去列的维度，得到每行合计。先手算再运行。


In [5]:
region = np.array([[10, 20], [30, 40], [50, 60]])
print("各地区三年累计：", region.sum(axis=0))  # [90, 120]
print("每年两地合计：", region.sum(axis=1))   # [30, 70, 110]
print("每地区三年平均：", region.mean(axis=0))


各地区三年累计： [ 90 120]
每年两地合计： [ 30  70 110]
每地区三年平均： [30. 40.]


## 06 比较与布尔筛选
比较得到True/False数组，再用它选取年份。多个数组条件用带括号的&，不是and。


In [6]:
demo_years = np.array([2021, 2022, 2023])
demo_yields = np.array([100.0, 120.0, 110.0])
above = demo_yields > demo_yields.mean()
print("条件：", above)
print("高于均值年份：", demo_years[above])
print("产量：", demo_yields[above])
print("100以上且不足120：", demo_yields[(demo_yields > 100) & (demo_yields < 120)])


条件： [False  True False]
高于均值年份： [2022]
产量： [120.]
100以上且不足120： [110.]


## 07 读取CSV并整理时间
输入data/粮食产量年度数据.csv，为教材原文件副本，UTF-8；原数据降序。只取完整的年份和粮食产量两列，其他列的空值不填0。replace去除“年”，astype把数字文本转数值。


In [7]:
# 文件有中文年份和表头，先按字符串只读前两列。
raw = np.loadtxt(DATA, delimiter=",", skiprows=1, usecols=(0, 1),
                 dtype=str, encoding="utf-8-sig")
year_list = []
for text in raw[:, 0]:
    year_list.append(int(text.replace("年", "")))
years_all = np.array(year_list)
yields_all = raw[:, 1].astype(float)
# 对年份排序后，产量必须用同一组位置重排，保持一一对应。
order = np.argsort(years_all)
years_all = years_all[order]
yields_all = yields_all[order]
print("形状：", raw.shape)
print("时间范围：", years_all[0], years_all[-1])
print("产量均为有限数：", np.all(np.isfinite(yields_all)))


形状： (75, 2)
时间范围： 1949 2023
产量均为有限数： True


## 08 选定本周统计范围
2014—2023年共10个年度观测，9个相邻年度变化。全历史为1949—2023年75行；“最近”仅相对此文件，不代表当前年份。


In [8]:
# 明确筛选年份，不依赖文件前10行恰好是最近10年。
mask = (years_all >= 2014) & (years_all <= 2023)
years = years_all[mask]
yields = yields_all[mask]
print("年份：", years)
print("产量（万吨）：", yields)


年份： [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]
产量（万吨）： [63964.83 66060.27 66043.51 66160.73 65789.22 66384.34 66949.15 68284.75
 68652.77 69541.  ]


## 09 描述统计与极值
累计是10年产量之和，不是某一年的产量。std的单位也是万吨，不是百分比；不把波动自动归因于政策或天气。argmax返回位置，若并列只返回首个位置，因此并列年份用布尔条件全部取出。


In [9]:
mean_yield = np.mean(yields)
std_yield = np.std(yields, ddof=0)  # 描述选定10年的总体离散程度，分母为10。
print(f"2014—2023年累计产量：{np.sum(yields):.2f} 万吨")
print(f"平均年产量：{mean_yield:.2f} 万吨")
print(f"总体标准差：{std_yield:.2f} 万吨")
print("最高年份：", years[yields == np.max(yields)], "产量：", np.max(yields))
print("最低年份：", years[yields == np.min(yields)], "产量：", np.min(yields))
print("首个最大值的位置：", np.argmax(yields))


2014—2023年累计产量：667830.57 万吨
平均年产量：66783.06 万吨
总体标准差：1546.95 万吨
最高年份： [2023] 产量： 69541.0
最低年份： [2014] 产量： 63964.83
首个最大值的位置： 9


## 10 高于均值与排名
排名用独立索引，不覆盖按时间排列的数组，否则增长计算就错了。


In [10]:
above = yields > mean_yield
print("高于均值年份：", years[above])
print("数量：", np.sum(above))
rank = np.argsort(yields)[::-1]
print("前5年份：", years[rank[:5]])
print("对应产量（万吨）：", yields[rank[:5]])


高于均值年份： [2020 2021 2022 2023]
数量： 4
前5年份： [2023 2022 2021 2020 2019]
对应产量（万吨）： [69541.   68652.77 68284.75 66949.15 66384.34]


## 11 相邻年增长分析
先用模拟[100,120,110]口算增长量[20,-10]，增长率[20%,-8.33%]。产量为0时不能直接作分母；本文件选定期内均为正数。


In [11]:
# 时间已经升序；今年在后，上年在前，分母必须是上年产量。
growth = yields[1:] - yields[:-1]
growth_rate = growth / yields[:-1] * 100
growth_years = years[1:]
print("对应年份：", growth_years)
print("年增长量（万吨）：", np.round(growth, 2))
print("同比增长率（%）：", np.round(growth_rate, 2))
print("下降年份：", growth_years[growth < 0])


对应年份： [2015 2016 2017 2018 2019 2020 2021 2022 2023]
年增长量（万吨）： [2095.44  -16.76  117.22 -371.51  595.12  564.81 1335.6   368.02  888.23]
同比增长率（%）： [ 3.28 -0.03  0.18 -0.56  0.9   0.85  1.99  0.54  1.29]
下降年份： [2016 2018]


## 12 规范导出分析结果
教师给定导出模板，不要求零基础学生背fmt。前一年不在窗口内，2014年同比在本任务中不计算，导出从2015年开始。写入仅在本包outputs内。


In [12]:
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)
result = np.column_stack((growth_years, yields[1:], growth, growth_rate))
np.savetxt(OUT / "2015至2023年增长分析.csv", result, delimiter=",",
           header="年份,粮食产量_万吨,年增长量_万吨,同比增长率_百分数",
           comments="", fmt=["%d", "%.2f", "%.2f", "%.4f"], encoding="utf-8-sig")
print("结果保存到：", OUT / "2015至2023年增长分析.csv")


结果保存到： /Users/lyb/Desktop/python金融大数据分析（课堂教学文件）/备课工作区/第03周_NumPy粮食产量分析/outputs/2015至2023年增长分析.csv


## 拓展A 随机数与矩阵概念
模拟播种面积单位千公顷，不是真实观测，不与真实产量混合推断单产。固定seed便于同学复现。二维ndarray足以完成本周任务；*逐元素相乘，@才是矩阵乘法。


In [13]:
rng = np.random.default_rng(42)
area_sim = np.round(rng.uniform(110000, 120000, size=10), 2)
print("模拟面积（千公顷）：", area_sim)
left = np.array([[1, 2], [3, 4]])
right = np.array([[2, 0], [0, 2]])
print("逐元素乘法：", left * right)
print("矩阵乘法：", left @ right)


模拟面积（千公顷）： [117739.56 114388.78 118585.98 116973.68 110941.77 119756.22 117611.4
 117860.64 111281.14 114503.86]
逐元素乘法： [[2 0]
 [0 8]]
矩阵乘法： [[2 4]
 [6 8]]


## 拓展B 全历史与时期比较
承接本校教案的历史对比，将末期统一为文件最新10年2014—2023。两个十年均值之比的增长率，不是同比也不是年均复合增长率。历史统计口径可变，不能仅据此推断因果。


In [14]:
early = yields_all[(years_all >= 1949) & (years_all <= 1958)]
late = yields_all[(years_all >= 2014) & (years_all <= 2023)]
print("最早10年平均年产量（万吨）：", early.mean())
print("最新10年平均年产量（万吨）：", late.mean())
print("两期平均水平增幅（%）：", (late.mean() / early.mean() - 1) * 100)
print("全75年累计产量（万吨）：", yields_all.sum())
print("全75年平均年产量（万吨）：", yields_all.mean())
# 2014到2023是9个年度间隔，不能用10作指数分母。
cagr = ((yields[-1] / yields[0]) ** (1 / (years[-1] - years[0])) - 1) * 100
print("2014—2023年复合年均增长率（%）：", cagr)


最早10年平均年产量（万吨）： 16587.12
最新10年平均年产量（万吨）： 66783.057
两期平均水平增幅（%）： 302.61996657647626
全75年累计产量（万吨）： 2880143.98
全75年平均年产量（万吨）： 38401.919733333336
2014—2023年复合年均增长率（%）： 0.9330269838975447
